# Stack 3 : Graph RAG — graphe d'entités + recherche vectorielle

Ce notebook construit un **graphe de connaissances** en mémoire (`networkx`) à
partir des entités extraites des chunks, puis fait une récupération **sensible au
graphe** : amorçage vectoriel (FAISS) + **traversée** des entités partagées.

- **Données / Chunking / Embeddings** : identiques aux Stacks 1 et 2.
- **Nouveauté** : extraction d'entités (heuristique) + graphe networkx + traversée.
- **Génération** : Ollama (`llama3.1`).
- Le code vit dans `src/stack3_graphrag/` — ce notebook l'**importe**.

## 1. Configuration

In [ ]:
import sys
import time
import json
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from shared.chunker import recursive_chunk, Chunk
from shared.embeddings import EmbeddingModel
from shared.llm import call_llm

print(f"Project root: {PROJECT_ROOT}")

## 2. Chargement des données

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikimedia/wikipedia", "20231101.simple", split="train[:100]")
print(f"{len(ds)} articles chargés")

## 3. Découpage (chunking)

In [ ]:
all_chunks: list[Chunk] = []
for doc in ds:
    all_chunks.extend(recursive_chunk(
        doc["text"], max_size=500, overlap=50,
        metadata={"title": doc["title"], "url": doc["url"]},
    ))

texts = [c.text for c in all_chunks]
metadata = [c.metadata for c in all_chunks]
print(f"{len(all_chunks)} chunks")

## 4. Embeddings + index FAISS

In [ ]:
from shared.vector_index import FaissIndexer

emb_model = EmbeddingModel("all-MiniLM-L6-v2")
indexer = FaissIndexer(dimension=emb_model.dimension)
indexer.add(emb_model.encode(texts), texts, metadata)
print(f"Index FAISS : {indexer.size} vecteurs")

## 5. Extraction d'entités + construction du graphe

On extrait les entités (noms propres) de chaque chunk, puis on construit un
graphe : un nœud par chunk et par entité, une arête `MENTIONS` entre eux, et
`RELATED_TO` entre entités co-occurrentes.

In [ ]:
from stack3_graphrag import extract_entities, build_graph

print("Entités du 1er chunk :", extract_entities(texts[0])[:8])

graph = build_graph(texts, metadata)
n_chunks = sum(1 for _, d in graph.nodes(data=True) if d.get("type") == "chunk")
n_entities = sum(1 for _, d in graph.nodes(data=True) if d.get("type") == "entity")
print(f"\nGraphe : {graph.number_of_nodes()} nœuds "
      f"({n_chunks} chunks, {n_entities} entités), {graph.number_of_edges()} arêtes")

In [ ]:
# Entités les plus mentionnées (degré le plus élevé)
entity_degrees = [
    (d["name"], graph.degree(n))
    for n, d in graph.nodes(data=True) if d.get("type") == "entity"
]
for name, deg in sorted(entity_degrees, key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {name}: {deg} liens")

## 6. Récupérateur de graphe

`GraphRetriever` amorce avec une recherche vectorielle, puis suit le motif
(chunk)-[:MENTIONS]->(entité)<-[:MENTIONS]-(voisin) pour récupérer des chunks
reliés par des entités partagées.

In [ ]:
from stack3_graphrag import GraphRetriever

graph_ret = GraphRetriever(indexer, emb_model, graph)

for i, r in enumerate(graph_ret.search("Which science studies plants?", k=5), 1):
    shared = r["metadata"].get("shared_entities")
    extra = f" — entités partagées : {', '.join(shared)}" if shared else ""
    print(f"{i}. (score={r['score']:.4f}) [{r['metadata'].get('title', '')}]{extra}")

## 7. Pipeline RAG graphe

`GraphRAG` réutilise le squelette RAG partagé : seule la récupération change.

In [ ]:
from stack3_graphrag import GraphRAG

rag = GraphRAG(graph_ret, llm_fn=call_llm)
print(rag.prompt_template)

In [ ]:
question = "What happened in April 1912?"
result = rag.query(question, k=5)

print(f"Question : {question}")
print(f"Réponse : {result['answer']}")
print(f"\nRécupération : {result['retrieval_ms']} ms")
print(f"Génération  : {result['generation_ms']} ms")
print(f"Total       : {result['latency_ms']} ms")

In [ ]:
test_queries = [
    "What happened in April 1912?",
    "Which science studies plants?",
    "Who invented the telephone?",
]
for q in test_queries:
    r = rag.query(q)
    print(f"Q : {q}")
    print(f"R : {r['answer']}")
    print(f"   [{r['latency_ms']} ms]\n")

## 8. Évaluation RAGAS

In [ ]:
eval_path = PROJECT_ROOT / "eval" / "questions.json"
if eval_path.exists():
    with open(eval_path, encoding="utf-8") as f:
        eval_data = json.load(f)
else:
    eval_data = [
        {"question": "What happened in April 1912?",
         "ground_truth": "The RMS Titanic sank after hitting an iceberg on April 15, 1912."},
        {"question": "Which science studies plants?",
         "ground_truth": "Botany is the science that studies plants."},
    ]
print(f"{len(eval_data)} questions d'évaluation")

In [ ]:
eval_questions = [e["question"] for e in eval_data]
eval_ground_truths = [e["ground_truth"] for e in eval_data]

eval_answers, eval_contexts = [], []
for q in eval_questions:
    r = rag.query(q)
    eval_answers.append(r["answer"])
    eval_contexts.append([c["text"] for c in r["contexts"]])
print(f"Réponses générées pour {len(eval_questions)} questions.")

In [ ]:
try:
    from shared.evaluator import evaluate_rag
    ragas_results = evaluate_rag(
        questions=eval_questions, answers=eval_answers,
        contexts=eval_contexts, ground_truths=eval_ground_truths,
    )
    print("RAGAS — Stack 3 (Graphe)")
    for m in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]:
        print(f"  {m}: {ragas_results.get(m, 'N/A')}")
    out = PROJECT_ROOT / "eval" / "graph_eval.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(ragas_results, f, indent=2, default=str)
    print(f"Résultats enregistrés : {out}")
except Exception as exc:
    print(f"Évaluation RAGAS impossible : {exc}")
    print("(RAGAS peut nécessiter une config supplémentaire, ex. un LLM juge.)")

## Résumé

Le Stack 3 (graphe) ajoute le **raisonnement relationnel** : il relie les chunks
par les entités qu'ils partagent. Utile pour les questions multi-hop, au prix de
la construction du graphe et d'une extraction d'entités de qualité.

Comme les autres stacks, seule la **récupération** diffère — tout le reste est
partagé (`src/shared/`).